# Collocations

Collocations are collections of words that occur together with greater than chance frequency. Identifying the collocations that a word participates in requires counting what falls near the word. This notebook works through that in two examples, once on a small tagged corpus where the whole result fits on a screen, and once on the BNC:

1. **Node word and context** -- finding a word's matches and taking a window around each one.
2. **The four frequencies** -- `collocates`, and the counts every association measure reads.
3. **Ranking** -- `collocations`, log-likelihood and log-Dice.
4. **Word and tag together** -- collocating on a struct, and cutting the result by part of speech.
5. **Two node words at once** -- the collocates that *time* and *money* share.

**References**

- Dunning, T. 1993. Accurate methods for the statistics of surprise and coincidence. *Computational Linguistics* 19: 61-74.
- Firth, J. R. 1957. A synopsis of linguistic theory 1930-1955. In *Studies in Linguistic Analysis*, 1-32. Oxford: Blackwell.
- Lakoff, G. and M. Johnson. 1980. *Metaphors We Live By*. Chicago: University of Chicago Press.
- Rychlý, P. 2008. A lexicographer-friendly association score. In *Proceedings of RASLAN*, 6-9. Brno: Masaryk University.

In [1]:
import polars as pl
import polars_corpus as plc
import nltk

pl.Config.set_tbl_rows(20)

polars.config.Config

## The corpus

`from_nltk` reads any corpus NLTK can supply into a DataFrame with one row per token. Brown is tagged, read as sentences and categorized, so it arrives with all five columns the reader can offer: `token`, `pos`, `sentence_tag`, `file_id` and `category`. The conversion walks the corpus in Python, which takes a moment; a corpus of any size is better read from parquet, as the BNC is below.

Two of those columns matter here beyond the words themselves. `pos` carries tags in the Brown tagset and `file_id` names the 500 texts. The corpus is 1,161,192 tokens, the figure that turns up as `n` in every table below.

In [2]:
brown = plc.from_nltk(nltk.corpus.brown)

## Choosing a node word

`search` takes a [simple, BNCweb-style query](../../simple_query/); a bare word is matched against the `token` column, and word-form matching is case-insensitive. What comes back is a `SearchResults` object, which holds the corpus along with the span each match covers rather than the words themselves, so its `repr` reports a count -- 333 matches for *light*. The words are read back out when `concordance` or `collocates` asks for them.

In [3]:
m = plc.search(brown, 'light')
m


SearchResults<'light'; 333 matches>

`concordance(window=3)` lays those matches out as KWIC lines, one row per match, with the match in `token` and three words on either side in `token_left_context` and `token_right_context`. (See [Concordances](../concordance/) for what that is good for.)

Reading down the lines, the node word is doing at least two jobs, the physical *light* of *light socket* and *bright light*, and the abstract one of *in the light of those events*.

In [4]:
m.concordance(window=3)

token_left_context,token,token_right_context
list[str],list[str],list[str]
"[""``"", ""In"", ""this""]","[""light""]","[""we"", ""need"", ""1,000""]"
"[""--"", ""``"", ""too""]","[""light""]","[""for"", ""me"", ""to""]"
"[""a"", ""$10,000"", ""morning""]","[""light""]","[""mink"", ""to"", ""Sportsman""]"
"[""height"", ""of"", ""the""]","[""light""]","[""socket"", "","", ""allowing""]"
"[""directly"", ""under"", ""the""]","[""light""]","[""."", ""Pulling"", ""strings""]"
"[""him"", ""from"", ""the""]","[""light""]","[""fixture"", ""by"", ""tying""]"
"[""."", ""In"", ""the""]","[""light""]","[""of"", ""those"", ""events""]"
"[""$250"", ""each"", "".""]","[""Light""]","[""reading"", "":"", ""Neither""]"
"[""--"", ""a"", ""bright""]","[""light""]","[""to"", ""lure"", ""other""]"


## Counting the context

`collocates` takes the same context the concordance just showed and counts it: every word in the window of every match, pooled across matches. Words occurring fewer than `min_freq` times in the windows are dropped -- the default of 5 is what leaves 52 of them here -- and the rows come back in no particular order, since nothing has been scored yet.

The `freqs` struct carries the four counts an association measure reads:

- `f12`, times the word fell in a window;
- `f1`, window positions available in total. Here that is 1998, which is 333 matches by the six positions a `window=3` gives each of them;
- `f2`, the word's frequency in the whole corpus;
- `n`, the corpus size, 1,161,192.

The matched tokens aren't counted. There is no `range` column: `collocates` counts occurrences, not the files they came from.

In [5]:
 m.collocates("token", window=3)

collocate,freqs
str,struct[4]
"""Company""","{6,1998,72,1161192}"
"""color""","{7,1998,137,1161192}"
"""of""","{105,1998,36080,1161192}"
"""an""","{5,1998,3542,1161192}"
"""this""","{8,1998,3966,1161192}"
"""when""","{5,1998,1746,1161192}"
"""incident""","{7,1998,49,1161192}"
"""The""","{21,1998,7258,1161192}"
"""so""","{5,1998,1755,1161192}"


## Ranking the collocates

[`collocations`](../../collocations/) does the same counting and then scores it. Three things change from the call above.

The window is the default 5 rather than 3, so `f1` rises to 3329 -- one short of 333 x 10, because one match sits four tokens from the start of its file and context never crosses a file boundary. The collocate is an expression rather than a column name, `pl.col("token").str.to_lowercase()`, which folds *The* in with *the* before the counting; anything that produces one column will do. And `min_range=2` drops the words whose window occurrences all come from a single text, which needs file ids to count -- they are there because `search` read `file_id` by default.

Asking for two measures gets a column for each, named for the measure rather than the argument (`LogLik`, `LogDice`), with the rows sorted by the first one asked for. Log-likelihood is Dunning's $G^2$, the distance between the observed counts and the ones independence predicts, signed so that a word occurring *less* often than expected comes out negative. Log-Dice weighs the joint frequency against the two marginals and ignores the corpus size altogether:

$$\text{logDice} = 14 + \log_2\frac{2\,f_{12}}{f_1 + f_2}$$

*the* leads on both counts, and not because it is distinctive: 297 of the 3329 context positions against the 201 its corpus frequency predicts is a small proportional excess over a very large count, which is enough for a $G^2$ of 43. Below it the list is the physical sense of the word -- *incident*, *source*, *visible*, *dark*, *bright* -- with *light* itself at f12 = 10, since a second occurrence of the node word inside a window is a collocate like any other.

In [6]:
colloc = plc.collocations(m, pl.col("token").str.to_lowercase(), ["ll", "logdice"], min_range=2)
colloc.head(10)

collocate,freqs,range,LogLik,LogDice
str,struct[4],u32,f64,f64
"""the""","{297,3329,69971,1161192}",115,43.441806,7.052794
"""incident""","{7,3329,49,1161192}",2,42.027375,6.085401
"""source""","{8,3329,94,1161192}",6,39.463697,6.258954
"""visible""","{6,3329,34,1161192}",3,38.737693,5.869429
"""dark""","{9,3329,185,1161192}",7,34.442524,6.391027
"""light""","{10,3329,333,1161192}",3,29.162155,6.483512
"""color""","{7,3329,140,1161192}",5,27.15597,6.047051
"""bright""","{5,3329,87,1161192}",5,20.751273,5.583836
"""power""","{8,3329,342,1161192}",3,19.707876,6.158043


### Word and tag together

`expr` may produce a struct, and then each distinct combination is its own collocate. `pl.struct(token, pos)` makes the word and its tag the unit, which is how a form's occurrences get divided among its readings, and the `collocate` column comes back as a `struct[2]` holding both.

Splitting counts cuts both ways. *incident* takes the top of the list, well above where it stood untagged: all seven of its appearances near *light* are the adjective, whose corpus frequency is 14 rather than the 49 of the form as a whole, so the same joint count is now measured against a much rarer word. *dark*, on the other hand, had nine window occurrences untagged and only five of them are the noun, which leaves the adjective reading below `min_freq` and out of the table entirely.

The tail is where the signed $G^2$ shows: *for*, *be*, *i*, *have* fall in these windows *less* often than their corpus frequencies predict, and negative `LogLik` is how that is reported. Sorting descending puts them last rather than first.

In [7]:
colloc = plc.collocations(m, pl.struct(pl.col("token").str.to_lowercase(), pl.col('pos')), ["ll", "logdice"], min_range=2)
colloc

collocate,freqs,range,LogLik,LogDice
struct[2],struct[4],u32,f64,f64
"{""incident"",""JJ""}","{7,3329,14,1161192}",2,62.610101,6.100427
"{""the"",""AT""}","{297,3329,69013,1161192}",115,46.319479,7.071773
"{""source"",""NN""}","{8,3329,90,1161192}",6,40.168683,6.260641
"{""visible"",""JJ""}","{6,3329,34,1161192}",3,38.737693,5.869429
"{""dark"",""NN""}","{5,3329,22,1161192}",4,35.068029,5.611552
"{""light"",""NN""}","{10,3329,253,1161192}",3,34.296339,6.515378
"{""color"",""NN""}","{7,3329,131,1161192}",5,28.059535,6.050799
"{""bright"",""JJ""}","{5,3329,77,1161192}",5,21.953878,5.588065
"{""window"",""NN""}","{5,3329,119,1161192}",3,17.722668,5.570384


The result is an ordinary DataFrame, so the tag is reachable with `struct.field` and the parts of speech can be taken one at a time. Brown's adjectives all begin with `J` and its nouns with `N`, which makes `starts_with` enough to gather a whole word class.

Between them the two frames are the sense split from the concordance, now sorted out: the adjectives are *incident*, *visible* and *bright*, and the nouns are *source*, *color*, *window* and *dark*, all of them the physical sense. The abstract *in the light of* leaves no content-word collocate strong enough to survive `min_freq`, only the *of* and *the* around it.

In [8]:
colloc.filter(pl.col('collocate').struct.field("pos").str.starts_with('J'))

collocate,freqs,range,LogLik,LogDice
struct[2],struct[4],u32,f64,f64
"{""incident"",""JJ""}","{7,3329,14,1161192}",2,62.610101,6.100427
"{""visible"",""JJ""}","{6,3329,34,1161192}",3,38.737693,5.869429
"{""bright"",""JJ""}","{5,3329,77,1161192}",5,21.953878,5.588065
"{""new"",""JJ""}","{6,3329,1056,1161192}",6,2.274475,5.486601


In [9]:
colloc.filter(pl.col('collocate').struct.field("pos").str.starts_with('N'))

collocate,freqs,range,LogLik,LogDice
struct[2],struct[4],u32,f64,f64
"{""source"",""NN""}","{8,3329,90,1161192}",6,40.168683,6.260641
"{""dark"",""NN""}","{5,3329,22,1161192}",4,35.068029,5.611552
"{""light"",""NN""}","{10,3329,253,1161192}",3,34.296339,6.515378
"{""color"",""NN""}","{7,3329,131,1161192}",5,28.059535,6.050799
"{""window"",""NN""}","{5,3329,119,1161192}",3,17.722668,5.570384
"{""room"",""NN""}","{5,3329,364,1161192}",5,7.803239,5.471351
"{""house"",""NN""}","{5,3329,388,1161192}",4,7.298083,5.462005
"{""night"",""NN""}","{5,3329,400,1161192}",3,7.060365,5.457355


No verb clears the cut at all. Brown spreads verbs across `VB`, `VBD`, `VBG`, `VBN` and `VBZ`, so a form's window occurrences are divided among up to five tags before `min_freq=5` is applied to each -- the same splitting that promoted *incident* here removes a word class. A coarser tagset, like the BNC's below, holds its counts together.

In [10]:
colloc.filter(pl.col('collocate').struct.field("pos").str.starts_with('V'))

collocate,freqs,range,LogLik,LogDice
struct[2],struct[4],u32,f64,f64


## Comparing two collocate lists

Lakoff and Johnson (1980) read **time is money** not as a figure of speech but as a conceptual metaphor that structures how the whole domain is talked about: time is spent, saved, wasted, run out of. If that is right, it should be visible in the collocates -- the two words should keep some of the same company. Below we take the collocates of each in the BNC and see which ones they share.

`pl.scan_parquet` gives a LazyFrame, and searching one gives `LazySearchResults`, which stores each match as an offset into its file and re-reads the corpus when it needs words. The 112-million-token BNC is never held in memory whole: the concordance reads back only the slices its matches fall in, and `collocations` makes one streaming pass over `lemma` and `pos` for the corpus totals. Reading only the columns named is what keeps that pass affordable.

`pl.struct("lemma", "pos")` collocates on the lemma and its tag, against the BNC's coarse tagset -- `SUBST`, `VERB`, `ADJ`, `ADV`, `PREP`, `PRON`, `ART`, `CONJ`, `STOP`. That last one is punctuation, which the BNC leaves unlemmatized, so all of it collects into the single `{null,"STOP"}` collocate; a corpus with one row per token has punctuation in the windows unless something takes it out. `min_freq` is raised to 100 to suit the size: `f1` is 1,524,921 context positions, about ten per match, out of `n` = 112,375,122 tokens.

Ranked by log-Dice, the top of the 1,104 rows is grammatical -- *at*, *the*, punctuation, *for*, *be*. Log-Dice does not divide by the corpus size, and with `f1` this large the frequent function words that fill a window at all are hard to displace. `with_row_index` numbers the rows in that order, which is what the join below needs.

In [11]:
bnc = pl.scan_parquet('../../data/bnc.parquet')

m = plc.search(bnc, 'time')
time = plc.collocations(m, pl.struct("lemma", "pos"), "logdice", window=5, min_freq=100).with_row_index()
time

index,collocate,freqs,range,LogDice
u32,struct[2],struct[4],u32,f64
0,"{""at"",""PREP""}","{35115,1524921,521623,112375122}",3438,9.135042
1,"{""the"",""ART""}","{103708,1524921,6040293,112375122}",3731,8.811218
2,"{null,""STOP""}","{175474,1524921,13606160,112375122}",3823,8.569886
3,"{""for"",""PREP""}","{25578,1524921,865253,112375122}",3280,8.453932
4,"{""be"",""VERB""}","{60178,1524921,4119764,112375122}",3648,8.448487
5,"{""a"",""ART""}","{36974,1524921,2134153,112375122}",3470,8.371176
6,"{""to"",""PREP""}","{38839,1524921,2593462,112375122}",3423,8.271572
7,"{""have"",""VERB""}","{23954,1524921,1316636,112375122}",3183,8.109729
8,"{""this"",""ADJ""}","{15880,1524921,452756,112375122}",2926,8.039548


The same call for *money*, with one difference: the cut is `min_range=100` rather than `min_freq=100`, so this list keeps the collocates found in at least 100 different files and leaves `min_freq` at its default of 5, where the *time* list kept collocates by window frequency and did not cut on range at all. That is why one list runs to 1,104 rows and the other to 249, and the two are not pruned on the same criterion.

The top is already more contentful than *time*'s was: *spend*, *get*, *pay* stand among the function words at ranks 1 through 8.

In [12]:
m = plc.search(bnc, 'money')
money = plc.collocations(m, pl.struct("lemma", "pos"), "logdice", window=5, min_range=100).with_row_index()
money

index,collocate,freqs,range,LogDice
u32,struct[2],struct[4],u32,f64
0,"{""for"",""PREP""}","{6276,365220,865253,112375122}",1855,7.384848
1,"{""spend"",""VERB""}","{1918,365220,22077,112375122}",958,7.342306
2,"{""get"",""VERB""}","{2455,365220,213376,112375122}",909,7.11931
3,"{""you"",""PRON""}","{4781,365220,805150,112375122}",1090,7.064563
4,"{""have"",""VERB""}","{6425,365220,1316636,112375122}",1726,6.967858
5,"{""to"",""PREP""}","{11301,365220,2593462,112375122}",2279,6.96764
6,"{""they"",""PRON""}","{4464,365220,842089,112375122}",1416,6.920758
7,"{""pay"",""VERB""}","{1372,365220,37398,112375122}",736,6.803013
8,"{""not"",""ADV""}","{3797,365220,767448,112375122}",1343,6.779351


### Shared collocates

An inner join on `collocate` keeps the lemma/tag pairs that appear in both lists -- 238 of *money*'s 249, so nearly everything that collocates with *money* also collocates with *time*. The join is on the struct column directly; structs compare field by field, so no unnesting is needed until the end, where `unnest()` opens `collocate` back into `lemma` and `pos`.

`index` and `index_right` are the two log-Dice ranks, and their mean, sorted ascending, ranks a collocate by how high it stands in both lists at once. Ranks rather than scores, because the two lists were cut at different depths and to different lengths, so a log-Dice value in one is not measured against the same pool as one in the other.

The bottom of the joint ranking is where the two words come apart: *bank*, *invest*, *worth*, *collect*, *pound* are all high on *money*'s list and far down *time*'s. The top is the shared grammar, and the metaphor is in between.

In [13]:
joint = time.join(money, on='collocate', how='inner').with_columns(((pl.col('index')+pl.col('index_right'))/2).alias('rank')).sort(by='rank', descending=False).select('rank','collocate').unnest()
joint

rank,lemma,pos
f64,str,str
1.5,"""for""","""PREP"""
5.0,"""the""","""ART"""
5.5,"""to""","""PREP"""
5.5,"""have""","""VERB"""
7.0,"""be""","""VERB"""
9.5,null,"""STOP"""
9.5,"""you""","""PRON"""
11.0,"""of""","""PREP"""
12.0,"""they""","""PRON"""


Cut to one word class at a time, the joint ranking is easier to read. The verbs at the top are the auxiliaries and light verbs that go with any noun, but from rank 18.5 down they are the metaphor's own vocabulary: *spend*, *get*, *make*, *give*, *take*. These are the verbs that construe their object as a quantity that can be transferred, exchanged and used up, and it is the same set for both nouns.

In [14]:
joint.filter(pl.col('pos')=='VERB').head(20)

rank,lemma,pos
f64,str,str
5.5,"""have""","""VERB"""
7.0,"""be""","""VERB"""
18.5,"""spend""","""VERB"""
21.5,"""get""","""VERB"""
29.0,"""will""","""VERB"""
30.0,"""do""","""VERB"""
35.5,"""make""","""VERB"""
43.5,"""would""","""VERB"""
45.0,"""give""","""VERB"""


Among the nouns, *time* and *money* each turn up in the other's windows -- the phrases that state the comparison outright. Then the quantity nouns, *lot*, *amount*, *waste*, which are what a mass noun of this kind takes: *a lot of*, *a waste of*, *the amount of*.

In [15]:
joint.filter(pl.col('pos')=='SUBST').head(20)

rank,lemma,pos
f64,str,str
42.0,"""time""","""SUBST"""
79.0,"""lot""","""SUBST"""
85.5,"""year""","""SUBST"""
86.5,"""people""","""SUBST"""
89.5,"""money""","""SUBST"""
116.5,"""amount""","""SUBST"""
140.0,"""way""","""SUBST"""
140.5,"""day""","""SUBST"""
149.5,"""week""","""SUBST"""


The BNC's `ADJ` covers the determiners and quantifiers as well as adjectives proper, and that is nearly all of what the two nouns share here: *that*, *some*, *all*, *this*, *any*, *more*, *much*, *little*, *most*. Both are mass nouns, so both take the same quantifier set, and quantification is exactly what the metaphor turns on. The evaluative adjectives -- *good*, *short*, *great* -- come further down.

In [16]:
joint.filter(pl.col('pos')=='ADJ').head(20)

rank,lemma,pos
f64,str,str
21.0,"""that""","""ADJ"""
31.0,"""some""","""ADJ"""
32.0,"""all""","""ADJ"""
33.0,"""this""","""ADJ"""
41.5,"""any""","""ADJ"""
44.0,"""more""","""ADJ"""
60.5,"""much""","""ADJ"""
63.5,"""good""","""ADJ"""
94.0,"""one""","""ADJ"""
